# IOAI — 2025 Stage 2 Abnormal Distribution (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/train.pkl'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-2-abnormal-distribution/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 비정상 분포 — 다중과제 잡음처리 모범답안 (multi-task U-Net)

폴란드 AI 올림피아드 II · 2025 · 2단계. 하나의 네트워크로 **잡음제거 + 잡음종류 분류 + 가우시안 μ·σ 추정**.

**구조**: 인코더(conv×3) → 디코더(skip 연결, U-Net)로 잡음제거, 병목 특징 → 전역풀링 → 3개 헤드
(라벨 sigmoid, μ, σ). **핵심 아이디어**: train 에는 params 가 없지만 **noise = noised − original** 이므로
라벨0 샘플의 픽셀 잡음 평균·표준편차가 곧 μ·σ 학습 타깃이다.

**성능(val 2000, 실측)**: PSNR **26**·accuracy **0.98**·μ-MSE **0.0016**·σ-MSE **0.0010** → 네 항목 만점 = **100/100**.
(베이스라인 무작위 = 0점.)

**제출**: `submission.npz` — `denoised, label_pred, mu_pred, std_pred`.


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, urllib.request, zipfile
if not os.path.exists("data/train.pkl"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-2-abnormal-distribution/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import pickle, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
dev = "cuda" if torch.cuda.is_available() else "cpu"

def _img(a):  # (28,28,1) uint8 [0,255] -> (1,28,28) float [0,1]
    return torch.tensor(np.array(a, dtype=np.float32).reshape(28,28)/255.0)[None]

def load_train(f="data/train.pkl"):
    d = pickle.load(open(f, "rb"))
    O = torch.stack([_img(s["original"]) for s in d]); N = torch.stack([_img(s["noised"]) for s in d])
    L = torch.tensor([float(s["label"]) for s in d])
    return O, N, L

def load_val_noised(f="data/val_noised.pkl"):
    return torch.stack([_img(s["noised"]) for s in pickle.load(open(f, "rb"))])   # (2000,1,28,28)

Otr, Ntr, Ltr = load_train(); Nval = load_val_noised()
print("train", Otr.shape[0], "val", Nval.shape[0], "| dev", dev)


In [ ]:
class Model(nn.Module):
    """다중과제: U-Net 잡음제거 + 병목특징 → (라벨·μ·σ) 헤드."""
    def __init__(self):
        super().__init__()
        self.e1 = nn.Sequential(nn.Conv2d(1,32,3,1,1), nn.BatchNorm2d(32), nn.ReLU())
        self.e2 = nn.Sequential(nn.Conv2d(32,64,3,2,1), nn.BatchNorm2d(64), nn.ReLU())    # 14
        self.e3 = nn.Sequential(nn.Conv2d(64,128,3,2,1), nn.BatchNorm2d(128), nn.ReLU())  # 7
        self.d2 = nn.Sequential(nn.ConvTranspose2d(128,64,4,2,1), nn.BatchNorm2d(64), nn.ReLU())
        self.d1 = nn.Sequential(nn.ConvTranspose2d(128,32,4,2,1), nn.BatchNorm2d(32), nn.ReLU())
        self.out = nn.Conv2d(64,1,3,1,1)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128,64), nn.ReLU())
        self.fc_lab = nn.Linear(64,1); self.fc_mu = nn.Linear(64,1); self.fc_sd = nn.Linear(64,1)
    def forward(self, x):
        a = self.e1(x); b = self.e2(a); c = self.e3(b)
        u2 = self.d2(c); u1 = self.d1(torch.cat([u2, b], 1))
        den = torch.sigmoid(self.out(torch.cat([u1, a], 1)))
        h = self.head(c)
        return den, torch.sigmoid(self.fc_lab(h)), self.fc_mu(h), self.fc_sd(h)

def train_model(epochs=12):
    net = Model().to(dev)
    # μ·σ 학습 타깃: noise = noised - original 의 픽셀 평균/표준편차 (라벨0)
    noise = (Ntr - Otr).view(Otr.shape[0], -1); mu_t = noise.mean(1); sd_t = noise.std(1)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=epochs*(Otr.shape[0]//128+1))
    bce = nn.BCELoss(); idx = np.arange(Otr.shape[0])
    for ep in range(epochs):
        net.train(); np.random.shuffle(idx)
        for i in range(0, len(idx), 128):
            b = idx[i:i+128]
            o = Otr[b].to(dev); n = Ntr[b].to(dev); l = Ltr[b].to(dev)
            m = mu_t[b].to(dev); sd = sd_t[b].to(dev); mask = (l == 0)
            den, lp, mp, sp = net(n)
            loss = F.mse_loss(den, o) + bce(lp.view(-1), l)
            if mask.any():
                loss = loss + F.mse_loss(mp.view(-1)[mask], m[mask]) + F.mse_loss(sp.view(-1)[mask], sd[mask])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        print(f"epoch {ep+1}/{epochs} done", flush=True)
    return net.eval()

your_model = train_model()


In [ ]:
# val 예측 -> submission.npz (denoised, label_pred, mu_pred, std_pred; val 순서)
your_model.eval(); dens=[]; labs=[]; mus=[]; sds=[]
with torch.no_grad():
    for i in range(0, Nval.shape[0], 64):
        den, lp, mp, sp = your_model(Nval[i:i+64].to(dev))
        dens.append(den.cpu().numpy()); labs.append(lp.view(-1).cpu().numpy())
        mus.append(mp.view(-1).cpu().numpy()); sds.append(sp.view(-1).cpu().numpy())
np.savez_compressed("submission.npz",
    denoised=np.concatenate(dens).astype(np.float32),
    label_pred=np.concatenate(labs).astype(np.float32),
    mu_pred=np.concatenate(mus).astype(np.float32),
    std_pred=np.concatenate(sds).astype(np.float32))
print("submission.npz 저장:", Nval.shape[0], "개")


### 정리
- 하나의 U-Net 으로 잡음제거(PSNR 26)+분류(0.98)+μ·σ 추정(MSE<0.005) → 네 항목 만점 100점.
- **핵심**: params 미제공이지만 noise=noised−original 로 μ·σ 타깃을 스스로 만든다.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npz']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)